# Dexamethasone response in human airway smooth muscle cells

**Question.** In GSE52778, which genes change expression after dexamethasone treatment (1 µM, 18 h) at a Benjamini–Hochberg adjusted p below 0.01 and an absolute log2 fold change of at least 1, once donor-to-donor differences are accounted for?

**Data.** Eight paired-end RNA-seq libraries from four primary airway smooth muscle cell lines (Himes et al., *PLoS One* 2014, PMID 24926665), each line sequenced untreated and treated. Counts come from `workflow/run_pipeline.sh` (HISAT2 to GRCh38, featureCounts on Ensembl genes).

**Model.** DESeq2 negative binomial GLM with design `~ cell + dex`. Including `cell` turns this into a paired comparison, which matters here because donor variation is larger than many treatment effects.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import pydeseq2
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "..")).resolve()
COUNTS_FILE = PROJECT_DIR / "results" / "counts" / "gene_counts.txt"
GENE_NAMES_FILE = PROJECT_DIR / "results" / "counts" / "gene_names.tsv"
SAMPLES_FILE = PROJECT_DIR / "config" / "samples.tsv"
OUT_DIR = PROJECT_DIR / "results" / "de"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PADJ_CUTOFF = 0.01   # BH-adjusted p threshold
LFC_CUTOFF = 1.0     # |log2 fold change| threshold (two-fold)
MIN_COUNT = 10       # pre-filter: count >= 10 in at least one full group of samples

print("pydeseq2", pydeseq2.__version__, "| pandas", pd.__version__)

## 1. Load counts and sample sheet

In [ ]:
fc = pd.read_csv(COUNTS_FILE, sep="\t", comment="#", index_col="Geneid")
counts = fc.iloc[:, 5:]  # drop Chr, Start, End, Strand, Length
counts.columns = [Path(c).name.removesuffix(".bam") for c in counts.columns]

samples = pd.read_csv(SAMPLES_FILE, sep="\t", index_col="run")
missing = sorted(set(samples.index) - set(counts.columns))
assert not missing, f"Samples in the sheet but not in the count matrix: {missing}"
counts = counts[samples.index].astype(int)

gene_names = pd.read_csv(GENE_NAMES_FILE, sep="\t", index_col="gene_id")["gene_name"]

print(f"{counts.shape[0]:,} genes x {counts.shape[1]} samples")
counts.sum().rename("assigned_fragments").to_frame().join(samples)

## 2. Filter low-count genes

Genes need at least `MIN_COUNT` fragments in as many samples as the smallest group, so a gene expressed in only one condition is kept.

In [ ]:
min_group = samples["dex"].value_counts().min()
keep = (counts >= MIN_COUNT).sum(axis=1) >= min_group
counts_f = counts.loc[keep]
print(f"Kept {keep.sum():,} of {len(keep):,} genes")

## 3. Fit DESeq2

In [ ]:
metadata = samples[["cell", "dex"]].copy()

dds = DeseqDataSet(
    counts=counts_f.T,          # pydeseq2 expects samples x genes
    metadata=metadata,
    design="~ cell + dex",
    refit_cooks=True,
    quiet=True,
)
dds.deseq2()

stats = DeseqStats(dds, contrast=["dex", "trt", "untrt"], alpha=PADJ_CUTOFF, quiet=True)
stats.summary()
res = stats.results_df.copy()
res.insert(0, "gene_name", gene_names.reindex(res.index).fillna(res.index.to_series()))
res.head()

## 4. Apply the thresholds from the question

In [ ]:
sig = res[(res["padj"] < PADJ_CUTOFF) & (res["log2FoldChange"].abs() >= LFC_CUTOFF)].copy()
sig["direction"] = np.where(sig["log2FoldChange"] > 0, "up", "down")
sig = sig.sort_values("padj")

summary = pd.Series({
    "genes tested": int(res["padj"].notna().sum()),
    "significant (padj < %.2g, |LFC| >= %.1f)" % (PADJ_CUTOFF, LFC_CUTOFF): len(sig),
    "up in dexamethasone": int((sig["direction"] == "up").sum()),
    "down in dexamethasone": int((sig["direction"] == "down").sum()),
})
res.to_csv(OUT_DIR / "deseq2_all_genes.csv")
sig.to_csv(OUT_DIR / "deseq2_significant.csv")
summary.to_csv(OUT_DIR / "summary.csv", header=["value"])
summary

In [ ]:
top = sig.head(20)[["gene_name", "baseMean", "log2FoldChange", "lfcSE", "padj", "direction"]]
top.round({"baseMean": 0, "log2FoldChange": 2, "lfcSE": 2})

## 5. Sanity check against known glucocorticoid targets

Before trusting the gene list, it should recover genes with well-documented glucocorticoid induction. *CRISPLD2* is the gene highlighted by the original study; *FKBP5*, *TSC22D3*, *PER1*, *ZBTB16* and *KLF15* are classic glucocorticoid receptor targets. If most of these are not significantly up, something upstream (sample labels, strandedness, annotation) is wrong.

In [ ]:
known_targets = ["CRISPLD2", "FKBP5", "TSC22D3", "PER1", "ZBTB16", "KLF15"]
check = res[res["gene_name"].isin(known_targets)][["gene_name", "log2FoldChange", "padj"]].copy()
check["recovered"] = (check["padj"] < PADJ_CUTOFF) & (check["log2FoldChange"] > 0)
n_found = int(check["recovered"].sum())
print(f"{n_found}/{len(known_targets)} known targets significantly up-regulated")
if n_found < len(known_targets) // 2:
    print("WARNING: fewer than half the expected targets recovered; review inputs before interpreting results.")
check.sort_values("padj")

## 6. Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Volcano
plot = res.dropna(subset=["padj"])
y = -np.log10(plot["padj"].clip(lower=1e-300))
is_sig = plot.index.isin(sig.index)
axes[0].scatter(plot["log2FoldChange"][~is_sig], y[~is_sig], s=4, c="0.75", rasterized=True)
axes[0].scatter(plot["log2FoldChange"][is_sig], y[is_sig], s=6, c="#c0392b", rasterized=True)
for g in sig.head(10).index:
    axes[0].annotate(res.at[g, "gene_name"], (res.at[g, "log2FoldChange"], -np.log10(max(res.at[g, "padj"], 1e-300))), fontsize=7)
axes[0].axhline(-np.log10(PADJ_CUTOFF), ls="--", lw=0.8, c="k")
for x in (-LFC_CUTOFF, LFC_CUTOFF):
    axes[0].axvline(x, ls="--", lw=0.8, c="k")
axes[0].set(xlabel="log2 fold change (dex / untreated)", ylabel="-log10 adjusted p", title="Volcano plot")

# MA plot
axes[1].scatter(np.log10(res["baseMean"] + 1)[~res.index.isin(sig.index)], res["log2FoldChange"][~res.index.isin(sig.index)], s=4, c="0.75", rasterized=True)
axes[1].scatter(np.log10(res.loc[sig.index, "baseMean"] + 1), res.loc[sig.index, "log2FoldChange"], s=6, c="#c0392b", rasterized=True)
axes[1].axhline(0, lw=0.8, c="k")
axes[1].set(xlabel="log10 mean normalised count", ylabel="log2 fold change", title="MA plot")

fig.tight_layout()
fig.savefig(OUT_DIR / "volcano_ma.png", dpi=200)

In [ ]:
# PCA on variance-stabilised counts: samples should separate by treatment and by donor
dds.vst(use_design=False)
vst = pd.DataFrame(dds.layers["vst_counts"], index=dds.obs_names, columns=dds.var_names)
top_var = vst.var().sort_values(ascending=False).index[:500]
X = vst[top_var] - vst[top_var].mean()
U, S, Vt = np.linalg.svd(X.values, full_matrices=False)
explained = S**2 / (S**2).sum()
pcs = pd.DataFrame(U[:, :2] * S[:2], index=X.index, columns=["PC1", "PC2"]).join(samples)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
markers = dict(zip(sorted(samples["cell"].unique()), "osD^"))
for (cell, dex), grp in pcs.groupby(["cell", "dex"]):
    ax.scatter(grp["PC1"], grp["PC2"], marker=markers[cell], s=60,
               c="#c0392b" if dex == "trt" else "#2c3e50", label=f"{cell} {dex}")
ax.set(xlabel=f"PC1 ({explained[0]:.0%})", ylabel=f"PC2 ({explained[1]:.0%})", title="PCA, top 500 variable genes")
ax.legend(fontsize=7, frameon=False, bbox_to_anchor=(1, 1))
fig.tight_layout()
fig.savefig(OUT_DIR / "pca.png", dpi=200)

## 7. Assumptions and limits

The thresholds (adjusted p below 0.01, two-fold change) are choices, not facts about the biology, so the exact gene count moves if either is changed; `deseq2_all_genes.csv` holds every tested gene for anyone wanting a different cut. FDR control at 1% means roughly one in a hundred listed genes is expected to be a false positive. Four donors is a small sample, and all were white male donors, so effects that vary by sex or ancestry cannot be seen here. These are cultured cells exposed for 18 hours, which says nothing direct about longer exposure or about airway tissue in patients. Fold changes are unshrunk Wald estimates; low-count genes can show large but unstable values, which is why `baseMean` is reported alongside them.

In [ ]:
print("Python", sys.version.split()[0])
for mod in (np, pd, matplotlib, pydeseq2):
    print(f"{mod.__name__:<12}{mod.__version__}")